# Real-model family robustness runs

Use a GPU runtime.

This notebook clones the repo, mounts Google Drive, runs smoke tests, then runs the full in-context retrieval probe for open-weight model families.

Outputs are written to `/content/drive/MyDrive/lengthgen_realmodel_family/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.isdir('/content/drive/MyDrive')
print('Drive mounted')


In [ ]:
!nvidia-smi
!python -m pip install -q --upgrade 'transformers>=4.45' accelerate sentencepiece protobuf


In [ ]:
%cd /content
!rm -rf lengthgen
!git clone https://github.com/arkankau/lengthgen.git
%cd /content/lengthgen
!git pull


In [ ]:
import os, subprocess, textwrap

OUTROOT = '/content/drive/MyDrive/lengthgen_realmodel_family'
os.makedirs(OUTROOT, exist_ok=True)

# Qwen is the highest-priority modern open-weight contrast.
# Gemma and Llama-family models may require accepting licenses or HF auth.
# TinyLlama is included as an ungated Llama-like backup.
MODELS = [
    ('qwen1p5b', 'Qwen/Qwen2.5-1.5B'),
    ('tinyllama1p1b', 'TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T'),
    ('gemma2b', 'google/gemma-2-2b'),
]

def run(cmd):
    print('\n' + '=' * 100)
    print(cmd)
    print('=' * 100)
    return subprocess.run(cmd, shell=True)


## Smoke tests

Smoke tests verify tokenizer compatibility, attention extraction, and short-run dynamic range.

Failures are recorded and skipped in the full run.


In [ ]:
working = []
for short, model in MODELS:
    outdir = f'{OUTROOT}/{short}_smoke'
    cmd = (
        f'python colab/real_model_probe.py '
        f'--model {model} --smoke --batch 2 '
        f'--outdir {outdir}'
    )
    result = run(cmd)
    if result.returncode == 0:
        working.append((short, model))
    else:
        print(f'SKIP full run for {model}; smoke failed with code {result.returncode}')

print('Working models:', working)


## Full runs

This uses conservative `--batch 2` because output attentions materialize full attention matrices.

If a model OOMs, rerun that model with `--batch 1`.


In [ ]:
for short, model in working:
    outdir = f'{OUTROOT}/{short}_h8'
    cmd = (
        f'python colab/real_model_probe.py '
        f'--model {model} '
        f'--lengths 5,10,20,40,80,160 '
        f'--n 150 --heads 8 --batch 2 '
        f'--outdir {outdir}'
    )
    run(cmd)


## Head-count robustness for Qwen

This is the main head-selection-artifact check.


In [ ]:
for heads in [4, 16]:
    outdir = f'{OUTROOT}/qwen1p5b_h{heads}'
    cmd = (
        f'python colab/real_model_probe.py '
        f'--model Qwen/Qwen2.5-1.5B '
        f'--lengths 5,10,20,40,80,160 '
        f'--n 150 --heads {heads} --batch 2 '
        f'--outdir {outdir}'
    )
    run(cmd)


## Package outputs

This copies result files into one directory and prints the file list for download or later sync.


In [ ]:
!mkdir -p "$OUTROOT/collected"
!find "$OUTROOT" -name realmodel_results.json -print
!python - <<'PY'
import os, shutil
root = '/content/drive/MyDrive/lengthgen_realmodel_family'
collected = os.path.join(root, 'collected')
os.makedirs(collected, exist_ok=True)
for dirpath, _, files in os.walk(root):
    if 'realmodel_results.json' not in files:
        continue
    name = os.path.basename(dirpath)
    shutil.copy2(os.path.join(dirpath, 'realmodel_results.json'), os.path.join(collected, f'realmodel_{name}.json'))
print('Collected files:')
for name in sorted(os.listdir(collected)):
    print(os.path.join(collected, name))
PY
